# Introduction and Dataset Description

## Introduction

This project studies how demographic structure (age and sex composition) relates to economic conditions across U.S. states, using **state-level population estimates** and **median household income** data. Because the income series is only available for a short window (2021–2023 in our collected dataset), our analysis emphasizes **cross-sectional comparisons across states** and **short-run changes** rather than long-horizon time-series modeling.

Specifically, we aim to:
- Compare income levels across states within each year
- Examine year-to-year changes in income and how unevenly states recover or slow down across 2021–2023
- Provide a clean, well-structured dataset that supports downstream EDA and feature engineering (e.g., age grouping, within-state deviations, growth rates)


## Dataset Description

### Data Sources (U.S. Census Bureau APIs)

We collected data from two official Census Bureau API products:

1. **Population Estimates Program (PEP) — Characteristics of the Population (CHarV)**
   - Endpoint: `https://api.census.gov/data/2023/pep/charv`
   - Purpose: provides annual **population counts by state, year, age, and sex**

2. **American Community Survey (ACS) 1-year estimates**
   - Endpoint pattern: `https://api.census.gov/data/{year}/acs/acs1`
   - Purpose: provides annual **median household income** by state
   - Variable used: `B19013_001E` (median household income in the past 12 months)

### Key Variables

After merging, the working dataset contains:
- `State_name` / `NAME`: state name  
- `State_code` / `STATE`: 2-digit FIPS code (as a string, zero-padded)  
- `Year` / `YEAR`: year (integer)  
- `Population` / `POP`: population count  
- `Age` / `AGE`: Census age code (includes “All ages” and grouped/single-year codes)  
- `Sex` / `SEX`: sex category code  
- `Median_household_income` / `median_hh_income`: ACS median household income  

### Analytical Time Window

The population endpoint includes multiple years, but the ACS median household income data were collected for **2021–2023**. To maintain consistent coverage across variables, the final analytic dataset focuses on **2021–2023** (rows outside this window that create missing income values are excluded during preprocessing).

---

# Data Acquisition Methodology

## Overview of Workflow

Data acquisition followed a reproducible API-based pipeline:

1. Send HTTP GET requests to the Census API endpoints  
2. Parse JSON responses into pandas DataFrames  
3. Clean and standardize key fields used for merging (state codes, year types)  
4. Concatenate yearly ACS income pulls (2021–2023) into one table  
5. Merge population and income tables on `(STATE, YEAR)`  
6. Export the merged dataset for downstream cleaning/EDA/feature engineering


## Population Data Collection (PEP CHarV)

We used the **PEP 2023 CHarV** endpoint:

- Endpoint: `https://api.census.gov/data/2023/pep/charv`
- Query parameters:
  - `get = NAME, STATE, YEAR, POP, AGE, SEX`
  - `for = state:*`

This returns a table of population estimates by state, year, age, and sex. The response is returned as JSON, where the first row contains column names and subsequent rows contain values. We convert it into a DataFrame using:

- `pd.DataFrame(data[1:], columns=data[0])`


## Median Household Income Collection (ACS 1-Year)

We collected state-level median household income from **ACS 1-year** for each year in `{2021, 2022, 2023}`:

- Endpoint pattern: `https://api.census.gov/data/{year}/acs/acs1`
- Query parameters:
  - `get = NAME, B19013_001E`
  - `for = state:*`

For each year:
- We request the data, parse the JSON, and convert it to a DataFrame.
- We add a `YEAR` column to label the observation year.
- We rename `B19013_001E` to a readable column name (e.g., `median_hh_income`).
- We convert `median_hh_income` to numeric with `errors="coerce"` to safely handle any non-numeric values.

Finally, we concatenate yearly tables into one ACS income dataset:

- `pd.concat([...], ignore_index=True)`


## Key Standardization Before Merge

To ensure a correct join across sources, we standardized:
- `STATE` as a **two-digit zero-padded string** using `.astype(str).str.zfill(2)`
- `YEAR` as **integer** using `.astype(int)`

This prevents mismatches such as `"1"` vs `"01"` and string vs integer year keys.


##  Dataset Merge and Export

We merged the population table with the ACS income table using:
- keys: `STATE`, `YEAR`
- join type: **left join** on the population dataset (to preserve demographic detail)

Example logic:
- `pop_df.merge(acs_income[["STATE","YEAR","median_hh_income"]], on=["STATE","YEAR"], how="left")`

The merged dataset was then exported to a CSV file (without index) for reproducibility:
- `merged_df.to_csv("merged_dataset.csv", index=False)`

This exported dataset is used in subsequent steps (cleaning, EDA, feature engineering, and reporting).


---

## Cleaning and Preprocessing Steps

After merging the population dataset with the ACS median household income dataset, we performed several cleaning and preprocessing steps to ensure consistency, remove redundancy, and prepare the data for analysis.

### 1. Initial Data Inspection
We first checked the basic structure of the merged dataset using `shape`, `columns`, and `describe(include='all')`. The merged dataset initially contained **90,480 rows and 8 columns**, including:
- `NAME`, `STATE`, `YEAR`
- `POP`, `AGE`, `SEX`
- `state` (duplicate column)
- `median_hh_income`

This inspection step helped identify naming inconsistencies, missing income values (for years not covered by ACS income), and redundant columns.

### 2. Column Renaming and Removing Redundancy
To improve readability and enforce consistent naming conventions, we renamed variables:
- `NAME` → `State_name`
- `STATE` → `State_code`
- `YEAR` → `Year`
- `POP` → `Population`
- `AGE` → `Age`
- `SEX` → `Sex`
- `median_hh_income` → `Median_household_income`

We also removed the duplicate `state` column to avoid confusion and redundancy.

### 3. Data Type Standardization
To ensure correct interpretation and enable reliable numerical operations, we standardized data types:
- `Year` was converted to integer
- `Population` and `Median_household_income` were converted to numeric
- `State_code` was formatted as a two-digit string using zero-padding (e.g., `"01"`)
- `State_name` and `Sex` were converted to categorical variables

These conversions ensure uniform formatting and make later grouping/aggregation steps more reliable.

### 4. Handling Missing Values via Year Filtering
Because ACS median household income is only available for **2021–2023**, the merged dataset contains missing income values for earlier years (e.g., 2020). Instead of imputing income, we restricted the dataset to years where income data exists:

- Filtered the dataset to include only `Year >= 2021`

After filtering, we ran a missing-value check and confirmed there were **no remaining missing values** in the cleaned dataset.

### 5. Removing Duplicate Rows
We removed any duplicated observations using `drop_duplicates()` and verified that:
- **0 duplicate rows** remained after cleaning.

### 6. Validating Income Structure at the State–Year Level
Finally, we validated that median household income is measured at the **state–year level** (not age- or sex-specific). We confirmed that within each `(State_code, Year)` pair, the income variable has **at most one unique value**. This matches the expected structure: income is constant across age/sex subgroups within the same state and year because it is merged onto demographic population records for analysis.

### Final Clean Dataset
The cleaned dataset:
- Includes only years **2021–2023**
- Has consistent column names and standardized data types
- Contains no missing values and no duplicate rows
- Preserves the correct state–year structure for median household income

This dataset is now ready for downstream EDA and feature engineering.


---

# Feature Engineering Process and Justification

This section describes the feature engineering steps applied to enhance interpretability, reduce noise, and improve cross-sectional comparability across states and years.



##  Age Code Transformation and Age Group Construction

### Filtering Valid Age Codes

The original `Age` variable contains detailed census-style age codes (e.g., `"0100"`, `"0200"`, `"8599"`), including aggregate categories:

- `"0000"` — All ages  
- `"8599"` — 85 years and over  

To ensure consistency and remove unnecessary granularity, we retained:

- All ages (`"0000"`)
- 85+ (`"8599"`)
- Single-year age groups ending in `"00"` (e.g., `"0100"`, `"0200"`, ..., `"8400"`)

This filtering step removes ambiguous or intermediate codes while preserving meaningful demographic structure.



### Converting Age Codes to Numeric Values

We converted valid age codes into numeric age values:

- `"0000"` → `NaN` (aggregate category)
- `"8599"` → `85` (representing 85+)
- `"XX00"` → `XX` (first two digits extracted)

This transformation allows age to be treated as a numeric variable for grouping and analysis.


### Creating Interpretable Age Labels

A readable `Age_label` variable was created:

- `"0000"` → `"All ages"`
- `"8599"` → `"85+"`
- Other ages → corresponding numeric age values as strings

This improves clarity in tables and visualizations.


### Constructing Broader Age Groups

To reduce excessive granularity and improve demographic interpretability, single-year ages were grouped into broader life-stage intervals:

- 0–18  
- 19–30  
- 31–45  
- 46–60  
- 61–75  
- 76–84  
- 85+  

The `"All ages"` category was retained separately for aggregate-level analysis.


#### Justification

Single-year age data may:

- Introduce noise in cross-sectional comparisons  
- Create sparsity in smaller population groups  
- Reduce clarity in visualization  

Grouping ages into meaningful intervals achieves:

- More stable population aggregation  
- Clearer demographic interpretation  
- Reduced sparsity  
- Improved usability in visualization and comparative analysis  

This transformation enhances structural clarity while preserving essential demographic patterns.

---

# 这个是我在code里面的顺序，我先把age的那个放到了最前面，然后把与national mean对比的两个feature加在了最后，你看看根据你加的那几个feature，按什么顺序排比较好，我觉得现在这样也ok
### Relative Income Measures

Median household income is measured at the state–year level. Direct comparisons across years may be influenced by national economic trends.

To improve interpretability, two additional income features were constructed.


### Income Relative to National Mean

For each year, the national average income was computed, and the following variable was defined:

Income Relative to National = State Income − National Mean Income (same year)

**Purpose:**

- Removes overall national time trends  
- Highlights each state's relative economic standing  
- Improves cross-sectional comparison within a year  


### Standardized Income (Z-score Within Year)

Income was also standardized within each year:

Z = (Income − Yearly Mean) / Yearly Standard Deviation

**Purpose:**

- Expresses income in standard deviation units  
- Enables scale-free comparison  
- Facilitates outlier detection  
- Improves interpretability in statistical modeling  

